# Text to SQL (Spider 2.0-Lite)

This example shows how to evaluate a `genlm.control` model on the [Spider 2.0-Lite](https://github.com/xlang-ai/Spider2) domain.

* **Task**: Generate SQL queries that answer complex analytical questions over real-world databases.
* **Data**: Spider 2.0-Lite (Lei et al., 2025). We focus on the locally executable SQLite subset (instances whose `instance_id` starts with `local`).

## Setup

First, install the dependencies for this domain. In the root directory, run:

```bash
pip install -e .[spider2]
```

To run the full Spider 2.0-Lite evaluation, clone the dataset:

```bash
git clone https://github.com/xlang-ai/Spider2.git
```

and follow the instructions in `Spider2/spider2-lite/README.md` to download the SQLite databases into
`Spider2/spider2-lite/resource/databases/spider2-localdb`. Pass that `spider2-lite` directory to
`Spider2Dataset.from_spider2_dir`.

For this example, we'll use the `assets/spider2/spider2_sample` directory which contains a small,
self-contained subset (one SQLite database, two questions) so the notebook is runnable as-is.

As in the original Spider notebook we also pass a `grammars.json` mapping schema names to lark
grammars; the same potential (`Spider2TableColumnVerifier`) used for Spider 1 is reused here.

In [ ]:
import os

os.environ["TOKENIZERS_PARALLELISM"] = "false"  # Avoid huggingface warnings

## Usage

This example shows how to evaluate a `genlm.control` model on Spider 2.0-Lite.

### Initialize the dataset and evaluator

In [ ]:
from genlm.eval.domains.spider2 import Spider2Dataset, Spider2Evaluator

In [ ]:
spider2_data_dir = "../../../assets/spider2/spider2_sample"  # Replace with your path to Spider 2.0-Lite
spider2_grammars = "../../../assets/spider2/spider2_sample/grammars.json"  # Replace with your grammars file

dataset = Spider2Dataset.from_spider2_dir(
    spider2_data_dir, grammar_json_path=spider2_grammars, few_shot_example_ids=[0, 1]
)

evaluator = Spider2Evaluator(spider2_data_dir)

### Define a model adaptor

A model adaptor is an async callable that takes a `Spider2Instance` and returns a `ModelOutput`. For this example, we'll use a constrained `genlm.control.PromptedLLM` to generate responses, just as in the Spider 1 notebook.

In [ ]:
from genlm.control import PromptedLLM, AWRS, BoolCFG
from genlm.eval import ModelOutput, ModelResponse
from genlm.eval.domains.spider2 import default_prompt_formatter

# Load an LLM
LLM = PromptedLLM.from_name("gpt2", eos_tokens=[b"\n", b"\n\n"])


async def model(instance, output_dir, replicate):
    # Set the prompt for the LLM.
    LLM.prompt_ids = default_prompt_formatter(
        LLM.model.tokenizer, instance, use_chat_format=False
    )

    # Use the schema-specific lark grammar to constrain generation.
    potential = BoolCFG.from_lark(instance.lark_grammar).coerce(LLM, f=b"".join)

    sampler = AWRS(LLM, potential)

    sequences = await sampler.smc(
        n_particles=2,
        ess_threshold=0.5,
        max_tokens=100,
    )

    return ModelOutput(
        responses=[
            ModelResponse(response=sequence, weight=prob)
            for sequence, prob in sequences.decoded_posterior.items()
        ],
    )

### Run the evaluation

The Spider 2.0-Lite evaluator executes the predicted SQL against the local SQLite database for the instance and compares it to the gold execution result(s) using the same pandas-table comparison as the official Spider 2.0-Lite evaluation suite.

In [ ]:
from genlm.eval import run_evaluation

results = await run_evaluation(
    dataset=dataset,
    model=model,
    evaluator=evaluator,
    max_instances=2,
    n_replicates=1,
    verbosity=1,
    # output_dir="spider2_results", optionally save the results to a directory
)

In [ ]:
results.keys()

In [ ]:
results["all_instance_outputs"]

## References

Fangyu Lei, Jixuan Chen, Yuxiao Ye, Ruisheng Cao, Dongchan Shin, Hongjin Su, Zhaoqing Suo,
Hongcheng Gao, Wenjing Hu, Pengcheng Yin, Victor Zhong, Caiming Xiong, Ruoxi Sun, Qian Liu,
Sida Wang, and Tao Yu. Spider 2.0: Evaluating language models on real-world enterprise text-to-SQL workflows.
In *International Conference on Learning Representations*, 2025. URL https://arxiv.org/abs/2411.07763.